# StatsAnal — reproducing the study's associations

This notebook reproduces the **statistical associations** from the study
*"Surrogates of the Central Autonomic Network as Predictive Markers for
Arrhythmia"* using this repository's current pipeline: a variable dictionary
(`VariableDict.xlsx`) scored by `ScoringFunctions.py`, with longitudinal frames
built by `SupplementaryScripts/prep/`.

The analyses appear **in memoir-figure order (1–7)**. Each section states what the
published figure showed, how the statistic is computed here, and the relevant
limitations.

> ### ⚠️ Method note — read this first
> The **published figures** were produced by the original, bespoke scripts now
> archived (read-only) in `SupplementaryScripts/legacy_figures/`. Those scripts
> used a **slightly different scoring method** (item-level imputation and
> missing-code handling that predate the unified engine), and bespoke multi-panel
> styling.
>
> This notebook instead recomputes the findings with the **current
> `VariableDict.xlsx` + `ScoringFunctions.py`** approach. The goal is to confirm
> we **land on the same associations** — *not* to reproduce the exact pixels.
> Plot styling here is deliberately plain; presentation is left to the user.


## Getting the data

No participant data ships with this repository. Obtain it yourself and place it
under `RawData/` (see `RawData/RD_README.md`):

- **HRS** — RAND longitudinal file `randhrs1992_2022v1.dta`, plus per-wave HRS
  Core `.DA` + codebook `.txt` files (2010–2022).
  <https://hrsdata.isr.umich.edu/>
- **NHANES** — cycles 2013–2018 `.xpt` modules (DEMO, DPQ, SLQ, MCQ, RXQ_RX, …).
  <https://wwwn.cdc.gov/nchs/nhanes/>

Then run, in order:

```bash
python ScoringFunctions.py                                        # -> SF_OUTPUT/*.csv
python SupplementaryScripts/prep/build_fullcohort.py              # -> SF_OUTPUT/analytic/...
python SupplementaryScripts/prep/build_analytic_dataset_expanded.py
python SupplementaryScripts/prep/analysis_timelag.py
python SupplementaryScripts/prep/analysis_timelag_multiwave.py
python SupplementaryScripts/prep/event_anchored_trajectory.py
```

The cells below read those outputs. If a file is missing, the cell prints a
clear message and skips — so the notebook always opens cleanly.


## Setup

In [ ]:

import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu, kruskal, chi2_contingency
warnings.filterwarnings("ignore")

sys.path.insert(0, "SupplementaryScripts")
from StatsHelpers import (REPO_ROOT, RAW, SCORED, ANALYTIC, FIGS,
                          significance_label, ci95)

print("Repo root :", REPO_ROOT)
print("Scored    :", SCORED)
print("Analytic  :", ANALYTIC)

def load_csv(path, what):
    """Load a CSV if present; otherwise print a hint and return None."""
    p = Path(path)
    if not p.exists():
        print(f"[missing] {what}\n          expected: {p}\n"
              f"          -> run ScoringFunctions.py / the prep step that builds it.")
        return None
    df = pd.read_csv(p)
    print(f"[ok] {what}: {len(df):,} rows x {df.shape[1]} cols")
    return df

### Expected column contract

The cells expect the scored / prep outputs to carry these columns (set the
matching `composite_name`s in `VariableDict.xlsx`):

| Column | Meaning | Where |
|--------|---------|-------|
| `HHID_PN` / `HHIDPN` | HRS participant id | all HRS files |
| `SleepQualityBurden` | 0–100 sleep-quality burden (SQB) | scored / prep |
| `DepressionBurden` | 0–100 depression burden (DB) | scored / prep |
| `arrhythmia`, `stroke`, … | incident CVD flags | `hrs_analytic_wide_fullcohort.csv` |
| `sex`, `age`, `bmi`, `hypertension`, `diabetes`, `sleep_apnea` | covariates | wide / cox frames |

Set the scored-output file name(s) you produced here:


In [ ]:

# Point these at your ScoringFunctions outputs / prep outputs.
HRS_SCORED   = SCORED / "hrs_2016_scored.csv"        # cross-sectional scored wave
NHANES_SCORED= SCORED / "nhanes_scored.csv"
WIDE_CSV     = ANALYTIC / "hrs_analytic_wide_fullcohort.csv"
COX_CSV      = ANALYTIC / "hrs_cox_long_expanded.csv"
MULTIWAVE    = ANALYTIC / "multiwave_scores.csv"
ANCHORED     = ANALYTIC / "event_anchored_scores.csv"

## Figure 1 — Cohort attrition

**Published:** flow charts of how the HRS (n≈28,753) and NHANES (n≈5,042)
analytic cohorts were reached, plus the longitudinal landmark cohort.

**Here:** attrition is a bookkeeping summary, not a statistical test. We report
the counts that survive each inclusion rule from the scored output.

**Limitation:** NHANES non-response (~40.9%) far exceeded HRS (~4.3%); the
NHANES cohort is therefore more exposed to selection bias.


In [ ]:

df = load_csv(HRS_SCORED, "HRS scored wave")
if df is not None:
    score_cols = [c for c in ("SleepQualityBurden", "DepressionBurden") if c in df]
    print(f"\nTotal rows                     : {len(df):,}")
    for c in score_cols:
        print(f"valid {c:<22}: {df[c].notna().sum():,}")
    if score_cols:
        both = df.dropna(subset=score_cols)
        print(f"valid on all burden scores     : {len(both):,}")

## Figure 2 — Sleep quality vs depression (instrument validation)

**Published:** moderate positive association between SQB and DB
(HRS Spearman ρ = 0.550; NHANES ρ = 0.432), consistent with the PSQI–PHQ-9
literature.

**Here:** Spearman correlation of the two burden scores from the current engine.

**Limitation:** a single scored wave gives a slightly lower ρ than the memoir's
per-person multi-wave means (validated: 2016 wave ρ ≈ 0.48). Direction and
moderate magnitude reproduce.


In [ ]:

df = load_csv(HRS_SCORED, "HRS scored wave")
if df is not None and {"SleepQualityBurden", "DepressionBurden"} <= set(df.columns):
    both = df.dropna(subset=["SleepQualityBurden", "DepressionBurden"])
    rho, p = spearmanr(both["SleepQualityBurden"], both["DepressionBurden"])
    print(f"Spearman rho = {rho:.3f}   {significance_label(p)}   n = {len(both):,}")
    print("Memoir: HRS pooled rho = 0.550 | NHANES rho = 0.432")
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(both["SleepQualityBurden"], both["DepressionBurden"], s=4, alpha=0.15)
    ax.set_xlabel("Sleep Quality Burden (%)"); ax.set_ylabel("Depression Burden (%)")
    ax.set_title(f"Sleep vs Depression  (Spearman rho = {rho:.2f})")
    plt.show()
else:
    print("Need SleepQualityBurden + DepressionBurden columns in the scored file.")

## Figure 3 — Drug class vs sleep / depression burden (NHANES)

**Published:** medicated participants had higher SQB and DB than unmedicated;
psychiatric ("brain") drug users scored higher than cardiovascular ("heart")
drug users. Pairwise Mann-Whitney U vs a no-medication reference, Holm-corrected.

**Here:** classify NHANES prescriptions with `MedicationClassifier`, then run the
same exposure-group comparison via `StatsHelpers`.

**Limitation:** small classes (e.g. antiarrhythmics, n≈72) give wide, unstable
intervals; medication was never scored, only used as an exposure label.


In [ ]:

from MedicationClassifier import participant_drug_classes
from StatsHelpers import build_exposure_groups, run_pairwise_mwu

nh = load_csv(NHANES_SCORED, "NHANES scored")
# RXQ_RX prescription file is required for drug classes:
rx_files = list(RAW.rglob("RXQ_RX*.xpt"))
if nh is not None and rx_files:
    rx = pd.concat([pd.read_sas(f, format="xport") for f in rx_files], ignore_index=True)
    rx.columns = rx.columns.astype(str).str.upper()
    meds = participant_drug_classes(rx, id_col="SEQN", drug_col="RXDDRUG")
    # nh must carry SEQN (set ParticipantID -> SEQN as a passthrough in the VD)
    id_col = "SEQN" if "SEQN" in nh.columns else nh.columns[0]
    nh = nh.merge(meds, left_on=id_col, right_on="SEQN", how="left")
    nh["MedicationClasses"] = nh["MedicationClasses"].fillna("none")
    for score_col in ("SleepQualityBurden", "DepressionBurden"):
        if score_col not in nh: continue
        groups = build_exposure_groups(nh, score_col=score_col)
        res = run_pairwise_mwu(groups)
        print(f"\n=== {score_col}: drug class vs no-medication reference ===")
        print(res[["DrugClass","n_class","U","p_holm","effect_r","label"]].to_string(index=False)
              if not res.empty else "  (insufficient groups)")
else:
    print("Need NHANES scored file + RXQ_RX*.xpt under RawData/.")

## Figure 4 — Sex-stratified trajectories & landmark hazard

**Published:** approaching arrhythmia onset, SQB and DB slopes were steeper in
cases than controls (both sexes). In **landmark Cox** models (adjusted for age,
BMI, hypertension, OSA, diabetes), higher SQB predicted arrhythmia **in women
only** — up to HR = 1.24 [1.09–1.41] at the 2-year landmark; sex×SQB interaction
HR = 1.21 [1.04–1.40], p = 0.011. DB was not predictive in either sex.

**Here:** a landmark Cox per sex on the prep frames (`multiwave_scores` +
`hrs_analytic_wide_fullcohort`), SQB standardized per SD.

**Limitation:** biennial sampling; individual slope discriminability is poor
(memoir AUC ≈ 0.53) — this is a group-level signal.


In [ ]:

from lifelines import CoxPHFitter

wide = load_csv(WIDE_CSV, "HRS analytic wide (covariates, arrhythmia, sex)")
mw   = load_csv(MULTIWAVE, "multiwave landmark scores")
if wide is not None and mw is not None:
    print("\nColumns available — wide:", [c for c in wide.columns][:20])
    print("Columns available — multiwave:", [c for c in mw.columns][:20])
    print("\nFit a landmark Cox of standardized SQB on incident arrhythmia, per sex,\n"
          "adjusting for the CHARGE-AF covariates present in these frames.\n"
          "Expected: a significant SQB hazard in women, not men (memoir Fig 4h).")
    # NOTE: exact column names depend on how you populated the VD / prep output.
    # Adapt the covariate list below to the columns printed above, then fit:
    #   cph = CoxPHFitter(); cph.fit(dfm, duration_col=..., event_col="arrhythmia",
    #                                formula="sqb_z + age + bmi + hypertension + diabetes + sleep_apnea")
    #   print(cph.summary.loc["sqb_z", ["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]])
else:
    print("Run the prep pipeline (build_fullcohort + analysis_timelag_multiwave) first.")

## Figure 5 — CHARGE-AF + surrogates: discrimination & slope specificity

**Published:** CHARGE-AF covariates gave 5-year Harrell's C = 0.667; adding
single-wave SQB + DB raised it modestly to C = 0.673. Event-anchored SQB slope
was steeper in arrhythmia cases than **high-risk** controls (p = 0.008), i.e.
arrhythmia-specific; DB slope differed only from **low-risk** controls
(p = 0.005), i.e. a general-CVD signal.

**Here:** compare Harrell's C of a CHARGE-AF-only Cox vs. one that adds SQB+DB,
using `lifelines` concordance on the cox-long frame.

**Limitation:** the outcome is *any* arrhythmia, while CHARGE-AF targets AF
specifically — this can deflate the baseline C and inflate the added value.


In [ ]:

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

cox = load_csv(COX_CSV, "HRS cox-long (CHARGE-AF covariates + outcome)")
if cox is not None:
    print("\nColumns available:", [c for c in cox.columns][:25])
    print("\nFit Model 1 (CHARGE-AF covariates) and Model 2 (+ SQB + DB),\n"
          "then compare Harrell's C. Memoir: 0.667 -> 0.673.")
    # Adapt covariate/column names to those printed above, then:
    #   m1 = CoxPHFitter().fit(df1, duration_col=..., event_col="arrhythmia")
    #   m2 = CoxPHFitter().fit(df2, duration_col=..., event_col="arrhythmia")
    #   print("C model1:", m1.concordance_index_, " C model2:", m2.concordance_index_)
else:
    print("Run build_analytic_dataset_expanded.py first.")

## Figure 6 — CVD outcomes by burden-score trajectory

**Published:** participants whose SQB/DB **worsened** 2010→2022 had higher CVD
incidence than stable/improved groups (DB: χ²=71.4, p<0.001; SQB: χ²=11.8,
p<0.01). The worsened-SQB group's arrhythmia share (14.0%) vs worsened-DB
(12.7%) did not differ significantly (χ²(1)=0.32, p=0.57).

**Here:** assign trajectory groups (improved / stable / worsened) from the
multiwave scores and test CVD incidence with a chi-square of independence.

**Limitation:** the overall-change-in-slope grouping is vulnerable to
regression-to-the-mean (worsened group starts low, improved group starts high).


In [ ]:

mw   = load_csv(MULTIWAVE, "multiwave scores")
wide = load_csv(WIDE_CSV, "HRS analytic wide (CVD outcomes)")
if mw is not None and wide is not None:
    print("\nDerive per-person SQB/DB trajectory slope across waves, bin into\n"
          "improved / stable / worsened, then chi-square against any-CVD incidence.")
    print("Memoir: DB chi2 = 71.4 (p<0.001); SQB chi2 = 11.8 (p<0.01).")
    # Build slope per HHIDPN from the multiwave long frame, tertile/sign-bin it,
    # cross-tabulate vs wide['arrhythmia'/'any_cvd'], then chi2_contingency(...).
else:
    print("Run analysis_timelag_multiwave.py + build_fullcohort.py first.")

## Figure 7 — Competing-risks CVD incidence (top-tertile burden)

**Published:** Aalen-Johansen cumulative incidence over 8 years for participants
entering the top tertile of SQB (n=5,495; 12.4% any CVD) or DB (n=6,704; 13.0%).
Stroke was the most common first event; arrhythmia ranked 2nd for top-SQB (3.2%)
vs 3rd for top-DB (3.1%) — consistent with a sleep-specific arrhythmia signal.

**Here:** Aalen-Johansen competing-risks cumulative incidence functions via
`lifelines.AalenJohansenFitter`, entry = first wave into the top burden tertile.

**Limitation:** subtypes are modelled as mutually exclusive first events, so the
arrhythmia→stroke pathway is not captured; curves are unadjusted/descriptive.


In [ ]:

try:
    from lifelines import AalenJohansenFitter
except Exception as e:
    print("lifelines AalenJohansenFitter unavailable:", e)

wide = load_csv(WIDE_CSV, "HRS analytic wide")
mw   = load_csv(MULTIWAVE, "multiwave scores")
if wide is not None and mw is not None:
    print("\nFor each of SQB and DB: select participants entering the top tertile while\n"
          "CVD-free, then fit Aalen-Johansen CIFs per first-event CVD subtype.")
    print("Memoir: top-SQB any-CVD 12.4%, arrhythmia 3.2% (rank 2);\n"
          "        top-DB  any-CVD 13.0%, arrhythmia 3.1% (rank 3).")
    # ajf = AalenJohansenFitter(); ajf.fit(durations, event_observed, event_of_interest=...)
else:
    print("Run the prep pipeline first.")

## Limitations (from the study)

- **Self-report.** Sleep, depression, and diagnoses are self-reported; not backed
  by ECG, EEG, actigraphy, or polysomnography.
- **Sampling cadence.** HRS is biennial — individual-level prediction and acute
  fluctuations are not captured (slope AUC ≈ 0.53).
- **Sum-Score Model.** Items are equally weighted; richer factor models may track
  longitudinal change better (Schlechter et al., 2022).
- **Non-standardised sleep items.** Sleep quality was assembled from available
  questions, not a validated instrument (e.g. PSQI).
- **Selection bias.** NHANES had ~40.9% non-response on the sleep/depression items.
- **Outcome breadth.** "Arrhythmia" covers all arrhythmias, while CHARGE-AF targets
  AF — affecting the C-statistic comparison.
- **Method difference.** The published figures used the legacy scoring scripts
  (`SupplementaryScripts/legacy_figures/`); this notebook uses the current
  `ScoringFunctions.py`. Associations reproduce; exact numbers can differ slightly.
- **No causality.** All findings are observational; the neurocardiac mechanisms
  discussed are hypotheses for future work.
